# Qwen: export a model and run a structured question on CPU

**You are here → 05 — Qwen with ExecuTorch.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| **→ 5. CPU portability (optional) (current)** | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| 6. Compatibility study (optional) | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

The serving chapters used an HTTP API. Here we package the same **Qwen3-4B** base revision as an **ExecuTorch XNNPACK** program and invoke its native CPU runner. The recipe uses **INT4 linear weights with dynamic INT8 activations and INT8 embeddings**, which differs from AWQ W4A16.

We test one direct clinical SQL question with a short schema. This is a portability smoke, separate from the full planning/repair workload and GPU serving benchmarks.

Configure the edge environment below, then run the cells in order. Reuse a validated export explicitly or perform conversion and export. **Generation always runs afresh in Live mode.** Install the [notebook environment](../SETUP.md). See [the edge setup guide](../tutorial/EXECUTORCH_MAC.md).


## 1. Imports and editable configuration

`MODE="live"` performs the stages below. Choose **plan** to inspect without edge execution or **replay** to read the explicitly selected prior record. The notebook mode overrides an old local JSON mode. No stage installs packages or downloads weights implicitly.


In [1]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.edge_workflow import EdgeStudy

MODE = "live"  # "plan" previews stages; "replay" displays archived results
MODE = os.environ.get('LOCAL_AI_LAB_MODE', MODE)  # Explicit CPU validation override.
# Machine settings: tutorial/config/edge-qwen.local.json
study = EdgeStudy('qwen', root=ROOT, mode=MODE)
print('Effective mode:', study.mode)
print('Model:', 'Qwen/Qwen3-4B')
print('Hardware target: local CPU / separate ExecuTorch environment')
print('Session:', study.session_id)
print('Current-session measurements:', len(study.results))
display(JSON(study.config))


Effective mode: live
Model: Qwen/Qwen3-4B
Hardware target: local CPU / separate ExecuTorch environment
Session: 20261110T121532-73d94917db
Current-session measurements: 0


<IPython.core.display.JSON object>

## 2. Preflight

Check the configured edge Python and installed runtime before any model work. Qwen requires a pinned source checkpoint and either a validated reusable export or explicit conversion/export. Nemotron needs only its pinned configuration. Missing prerequisites stop with instructions.


In [2]:
display(JSON(study.preflight()))


<IPython.core.display.JSON object>

## 3. Prepare and validate the CPU program

Stages run as separate processes so large reference and converted models do not stay resident together. `download_models=false` uses staged source weights. Set it to true only to request a download.

With `reuse_export=true`, conversion, reference, validation and export cells verify the recorded artifact hashes and report **reused_artifact**. They do not claim a new export. To build afresh, set `reuse_export=false` and select a new `artifact_dir` in the local config. Conversion uses the official Qwen3 weight mapping; FP32 logit checks must pass before quantized export.

The native recipe and generation call are in [run_qwen_edge.py](../scripts/tutorial/run_qwen_edge.py).


In [3]:
stage_result = study.stage('download')
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(stage_result)))


Starting edge stage: download | session 20261110T121532-73d94917db


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| download | reused_artifact | — | — | — | — | — | — | — |

| Session id | Session evidence path | Reason |
|---|---|---|
| 20261110T121532-73d94917db | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/download-9d9f59a0.json | Download disabled; using explicit local source |

<details><summary>Full result details</summary>

<pre>{
  &quot;measured_at&quot;: &quot;2026-11-10T12:15:33.178059+00:00&quot;,
  &quot;mode&quot;: &quot;live&quot;,
  &quot;reason&quot;: &quot;Download disabled; using explicit local source&quot;,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/download-9d9f59a0.json&quot;,
  &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;stage&quot;: &quot;download&quot;,
  &quot;status&quot;: &quot;reused_artifact&quot;
}</pre>
</details>

In [4]:
stage_result = study.stage('prepare')
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(stage_result)))


Starting edge stage: prepare | session 20261110T121532-73d94917db


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| prepare | reused_artifact | — | — | — | — | — | — | — |

| Session id | Session evidence path | Reason |
|---|---|---|
| 20261110T121532-73d94917db | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/prepare-1d98828c.json | Explicit reuse of hash-verified export and conversion validation; no new export claimed |

<details><summary>Full result details</summary>

<pre>{
  &quot;artifact_lineage&quot;: {
    &quot;context&quot;: 4096,
    &quot;files_sha256&quot;: {
      &quot;export.yaml&quot;: &quot;c7ad1920f881f6e8e04237d7f9cfbe5a47d443bddcb152c24f4a6cd01d42f175&quot;,
      &quot;params.json&quot;: &quot;23c89697ae2039c085fe028d067199acd53c067605d40b1194a4ba8ce56b6904&quot;,
      &quot;qwen3-4b-xnnpack.pte&quot;: &quot;9555592b7b3f28b79e059c9a72d2600eb36c9b5e43cdef73b7d8922c695810b0&quot;,
      &quot;qwen3-4b.pth&quot;: &quot;4fcf55ffe74047a86cfad9fa6f2357f745a9c5b91fdc7023d187966b7d6c22b4&quot;,
      &quot;validation.json&quot;: &quot;0f8572d50c33e5613a6e9c58e9801f456294480a958108f98f7d1e12268637d2&quot;
    },
    &quot;model_id&quot;: &quot;Qwen/Qwen3-4B&quot;,
    &quot;model_revision&quot;: &quot;1cfa9a7208912126459214e8b04321603b3df60c&quot;,
    &quot;qmode&quot;: &quot;8da4w&quot;
  },
  &quot;measured_at&quot;: &quot;2026-11-10T12:15:37.125470+00:00&quot;,
  &quot;mode&quot;: &quot;live&quot;,
  &quot;reason&quot;: &quot;Explicit reuse of hash-verified export and conversion validation; no new export claimed&quot;,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/prepare-1d98828c.json&quot;,
  &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;stage&quot;: &quot;prepare&quot;,
  &quot;status&quot;: &quot;reused_artifact&quot;
}</pre>
</details>

In [5]:
stage_result = study.stage('reference')
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(stage_result)))


Starting edge stage: reference | session 20261110T121532-73d94917db


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| reference | reused_artifact | — | — | — | — | — | — | — |

| Session id | Session evidence path | Reason |
|---|---|---|
| 20261110T121532-73d94917db | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/reference-a29d110a.json | Explicit reuse of hash-verified export and conversion validation; no new export claimed |

<details><summary>Full result details</summary>

<pre>{
  &quot;artifact_lineage&quot;: {
    &quot;context&quot;: 4096,
    &quot;files_sha256&quot;: {
      &quot;export.yaml&quot;: &quot;c7ad1920f881f6e8e04237d7f9cfbe5a47d443bddcb152c24f4a6cd01d42f175&quot;,
      &quot;params.json&quot;: &quot;23c89697ae2039c085fe028d067199acd53c067605d40b1194a4ba8ce56b6904&quot;,
      &quot;qwen3-4b-xnnpack.pte&quot;: &quot;9555592b7b3f28b79e059c9a72d2600eb36c9b5e43cdef73b7d8922c695810b0&quot;,
      &quot;qwen3-4b.pth&quot;: &quot;4fcf55ffe74047a86cfad9fa6f2357f745a9c5b91fdc7023d187966b7d6c22b4&quot;,
      &quot;validation.json&quot;: &quot;0f8572d50c33e5613a6e9c58e9801f456294480a958108f98f7d1e12268637d2&quot;
    },
    &quot;model_id&quot;: &quot;Qwen/Qwen3-4B&quot;,
    &quot;model_revision&quot;: &quot;1cfa9a7208912126459214e8b04321603b3df60c&quot;,
    &quot;qmode&quot;: &quot;8da4w&quot;
  },
  &quot;measured_at&quot;: &quot;2026-11-10T12:15:37.129905+00:00&quot;,
  &quot;mode&quot;: &quot;live&quot;,
  &quot;reason&quot;: &quot;Explicit reuse of hash-verified export and conversion validation; no new export claimed&quot;,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/reference-a29d110a.json&quot;,
  &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;stage&quot;: &quot;reference&quot;,
  &quot;status&quot;: &quot;reused_artifact&quot;
}</pre>
</details>

In [6]:
stage_result = study.stage('validate')
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(stage_result)))


Starting edge stage: validate | session 20261110T121532-73d94917db


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| validate | reused_artifact | — | — | — | — | — | — | — |

| Session id | Session evidence path | Reason |
|---|---|---|
| 20261110T121532-73d94917db | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/validate-b6d0a0cb.json | Explicit reuse of hash-verified export and conversion validation; no new export claimed |

<details><summary>Full result details</summary>

<pre>{
  &quot;artifact_lineage&quot;: {
    &quot;context&quot;: 4096,
    &quot;files_sha256&quot;: {
      &quot;export.yaml&quot;: &quot;c7ad1920f881f6e8e04237d7f9cfbe5a47d443bddcb152c24f4a6cd01d42f175&quot;,
      &quot;params.json&quot;: &quot;23c89697ae2039c085fe028d067199acd53c067605d40b1194a4ba8ce56b6904&quot;,
      &quot;qwen3-4b-xnnpack.pte&quot;: &quot;9555592b7b3f28b79e059c9a72d2600eb36c9b5e43cdef73b7d8922c695810b0&quot;,
      &quot;qwen3-4b.pth&quot;: &quot;4fcf55ffe74047a86cfad9fa6f2357f745a9c5b91fdc7023d187966b7d6c22b4&quot;,
      &quot;validation.json&quot;: &quot;0f8572d50c33e5613a6e9c58e9801f456294480a958108f98f7d1e12268637d2&quot;
    },
    &quot;model_id&quot;: &quot;Qwen/Qwen3-4B&quot;,
    &quot;model_revision&quot;: &quot;1cfa9a7208912126459214e8b04321603b3df60c&quot;,
    &quot;qmode&quot;: &quot;8da4w&quot;
  },
  &quot;measured_at&quot;: &quot;2026-11-10T12:15:37.133519+00:00&quot;,
  &quot;mode&quot;: &quot;live&quot;,
  &quot;reason&quot;: &quot;Explicit reuse of hash-verified export and conversion validation; no new export claimed&quot;,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/validate-b6d0a0cb.json&quot;,
  &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;stage&quot;: &quot;validate&quot;,
  &quot;status&quot;: &quot;reused_artifact&quot;
}</pre>
</details>

In [7]:
stage_result = study.stage('export')
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(stage_result)))


Starting edge stage: export | session 20261110T121532-73d94917db


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| export | reused_artifact | — | — | — | — | — | — | — |

| Session id | Session evidence path | Reason |
|---|---|---|
| 20261110T121532-73d94917db | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/export-69487e6c.json | Explicit reuse of hash-verified export and conversion validation; no new export claimed |

<details><summary>Full result details</summary>

<pre>{
  &quot;artifact_lineage&quot;: {
    &quot;context&quot;: 4096,
    &quot;files_sha256&quot;: {
      &quot;export.yaml&quot;: &quot;c7ad1920f881f6e8e04237d7f9cfbe5a47d443bddcb152c24f4a6cd01d42f175&quot;,
      &quot;params.json&quot;: &quot;23c89697ae2039c085fe028d067199acd53c067605d40b1194a4ba8ce56b6904&quot;,
      &quot;qwen3-4b-xnnpack.pte&quot;: &quot;9555592b7b3f28b79e059c9a72d2600eb36c9b5e43cdef73b7d8922c695810b0&quot;,
      &quot;qwen3-4b.pth&quot;: &quot;4fcf55ffe74047a86cfad9fa6f2357f745a9c5b91fdc7023d187966b7d6c22b4&quot;,
      &quot;validation.json&quot;: &quot;0f8572d50c33e5613a6e9c58e9801f456294480a958108f98f7d1e12268637d2&quot;
    },
    &quot;model_id&quot;: &quot;Qwen/Qwen3-4B&quot;,
    &quot;model_revision&quot;: &quot;1cfa9a7208912126459214e8b04321603b3df60c&quot;,
    &quot;qmode&quot;: &quot;8da4w&quot;
  },
  &quot;measured_at&quot;: &quot;2026-11-10T12:15:37.136798+00:00&quot;,
  &quot;mode&quot;: &quot;live&quot;,
  &quot;reason&quot;: &quot;Explicit reuse of hash-verified export and conversion validation; no new export claimed&quot;,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/export-69487e6c.json&quot;,
  &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;stage&quot;: &quot;export&quot;,
  &quot;status&quot;: &quot;reused_artifact&quot;
}</pre>
</details>

## 4. Invoke the model and check its answer

The next cell starts a **new native generation process** with the verified `.pte` program and tokenizer. The runner calls `TextLLMRunner.generate()` with temperature zero, a short clinical schema and an output cap of 192 tokens. It uses no HTTP serving API and loads no saved answer. Model-load time is separate from inference timing.

The following cell executes the newly generated SQL in DuckDB and compares its rows with the reference. SQL scoring occurs after native timing.


In [8]:
edge = study.stage('run')
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(edge)))


Starting edge stage: run | session 20261110T121532-73d94917db


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| run | completed | — | — | 0.224 | 0.990 | 1.632 | 4.908 | 2365.898 |

Memory scope: whole runner-process RSS, including Python/runtime imports; not GPU memory.

| Session id | Session evidence path |
|---|---|
| 20261110T121532-73d94917db | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-c0fa578c.json |

<details><summary>Full result details</summary>

<pre>{
  &quot;artifact_lineage&quot;: {
    &quot;context&quot;: 4096,
    &quot;files_sha256&quot;: {
      &quot;export.yaml&quot;: &quot;c7ad1920f881f6e8e04237d7f9cfbe5a47d443bddcb152c24f4a6cd01d42f175&quot;,
      &quot;params.json&quot;: &quot;23c89697ae2039c085fe028d067199acd53c067605d40b1194a4ba8ce56b6904&quot;,
      &quot;qwen3-4b-xnnpack.pte&quot;: &quot;9555592b7b3f28b79e059c9a72d2600eb36c9b5e43cdef73b7d8922c695810b0&quot;,
      &quot;qwen3-4b.pth&quot;: &quot;4fcf55ffe74047a86cfad9fa6f2357f745a9c5b91fdc7023d187966b7d6c22b4&quot;,
      &quot;validation.json&quot;: &quot;0f8572d50c33e5613a6e9c58e9801f456294480a958108f98f7d1e12268637d2&quot;
    },
    &quot;model_id&quot;: &quot;Qwen/Qwen3-4B&quot;,
    &quot;model_revision&quot;: &quot;1cfa9a7208912126459214e8b04321603b3df60c&quot;,
    &quot;qmode&quot;: &quot;8da4w&quot;
  },
  &quot;backend&quot;: &quot;XNNPACK / CPU&quot;,
  &quot;checkpoint_validation&quot;: {
    &quot;atol&quot;: 0.001,
    &quot;dtype&quot;: &quot;float32&quot;,
    &quot;input_sequences&quot;: 2,
    &quot;max_abs_logit_error&quot;: 4.482269287109375e-05,
    &quot;passed&quot;: true,
    &quot;rtol&quot;: 0.001,
    &quot;scope&quot;: &quot;Unquantized mapped model vs original HF; does not validate quantized clinical task quality.&quot;
  },
  &quot;command&quot;: [
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial/iterations/mvp3/publish/clinical-trial-mvp3-executed/.venv-edge/bin/python&quot;,
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial-mvp4 optimization/scripts/tutorial/run_qwen_edge.py&quot;,
    &quot;--stage&quot;,
    &quot;run&quot;,
    &quot;--folder&quot;,
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial-mvp4 optimization/models/edge-qwen3-4b&quot;,
    &quot;--output&quot;,
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial-mvp4 optimization/experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-raw-5d7cafda.json&quot;
  ],
  &quot;context&quot;: 4096,
  &quot;conversion&quot;: {
    &quot;converted_keys&quot;: 399,
    &quot;converter&quot;: &quot;executorch.examples.models.qwen3.convert_weights.qwen_3_tune_to_meta&quot;,
    &quot;converter_sha256&quot;: &quot;d8201eb745ed0e0ffde3fea6ae2f5d06d2867a05d1b6e22653ab470ed3b9bbdb&quot;,
    &quot;model_id&quot;: &quot;Qwen/Qwen3-4B&quot;,
    &quot;note&quot;: &quot;Official Qwen3 mapping preserves Q/K normalization and tied output embeddings.&quot;,
    &quot;revision&quot;: &quot;1cfa9a7208912126459214e8b04321603b3df60c&quot;,
    &quot;source_keys&quot;: 398
  },
  &quot;dtype&quot;: &quot;INT4 linear weights / dynamic INT8 activations&quot;,
  &quot;evidence&quot;: &quot;hardware&quot;,
  &quot;footprint_bytes&quot;: 2480823486,
  &quot;generation&quot;: {
    &quot;num_generated_tokens&quot;: 29,
    &quot;num_prompt_tokens&quot;: 52,
    &quot;observer&quot;: {
      &quot;SCALING_FACTOR_UNITS_PER_SECOND&quot;: 1000,
      &quot;aggregate_model_execution_time_ms&quot;: 981,
      &quot;aggregate_sampling_time_ms&quot;: 8,
      &quot;decode_token_per_sec&quot;: 37.859,
      &quot;first_token_ms&quot;: 1794313224650,
      &quot;generated_tokens&quot;: 29,
      &quot;inference_end_ms&quot;: 1794313225416,
      &quot;inference_start_ms&quot;: 1794313224426,
      &quot;model_execution_end_ms&quot;: 1794313225416,
      &quot;model_execution_start_ms&quot;: 1794313225391,
      &quot;model_load_end_ms&quot;: 1794313224426,
      &quot;model_load_start_ms&quot;: 1794313222794,
      &quot;prefill_token_per_sec&quot;: 232.143,
      &quot;prompt_eval_end_ms&quot;: 1794313224650,
      &quot;prompt_tokens&quot;: 52
    },
    &quot;prompt&quot;: &quot;&lt;|im_start|&gt;user\nWrite only DuckDB SQL, without Markdown or explanation.\nTable subjects(subject_id VARCHAR PRIMARY KEY, treatment_arm VARCHAR, screening_status VARCHAR).\nHow many subjects are Enrolled or Randomized by treatment arm?&lt;|im_end|&gt;\n&lt;|im_start|&gt;assistant\n&lt;think&gt;\n\n&lt;/think&gt;\n\n&quot;,
    &quot;question&quot;: &quot;How many subjects are Enrolled or Randomized by treatment arm?&quot;,
    &quot;text&quot;: &quot;SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm;&lt;|im_end|&gt;&quot;
  },
  &quot;hardware&quot;: &quot;Apple arm64 / CPU XNNPACK&quot;,
  &quot;limitation&quot;: &quot;One short-schema direct SQL portability smoke; not the full plan/repair workload, not a latency benchmark; file checks warm the filesystem cache. Native tokens retained as reported; operator fallback unreviewed.&quot;,
  &quot;measured_at&quot;: &quot;2026-11-10T12:20:26.784816+00:00&quot;,
  &quot;memory&quot;: {
    &quot;error&quot;: null,
    &quot;peak_bytes&quot;: 5269995520,
    &quot;sample_count&quot;: 297,
    &quot;sampling_interval_seconds&quot;: 0.1,
    &quot;scope&quot;: &quot;whole runner-process RSS, including Python/runtime imports; not GPU memory&quot;
  },
  &quot;mode&quot;: &quot;live&quot;,
  &quot;model_id&quot;: &quot;Qwen/Qwen3-4B&quot;,
  &quot;model_name&quot;: &quot;Qwen/Qwen3-4B&quot;,
  &quot;model_revision&quot;: &quot;1cfa9a7208912126459214e8b04321603b3df60c&quot;,
  &quot;model_sha256&quot;: &quot;9555592b7b3f28b79e059c9a72d2600eb36c9b5e43cdef73b7d8922c695810b0&quot;,
  &quot;native_observer&quot;: {
    &quot;SCALING_FACTOR_UNITS_PER_SECOND&quot;: 1000,
    &quot;aggregate_model_execution_time_ms&quot;: 981,
    &quot;aggregate_sampling_time_ms&quot;: 8,
    &quot;decode_token_per_sec&quot;: 37.859,
    &quot;first_token_ms&quot;: 1794313224650,
    &quot;generated_tokens&quot;: 29,
    &quot;inference_end_ms&quot;: 1794313225416,
    &quot;inference_start_ms&quot;: 1794313224426,
    &quot;model_execution_end_ms&quot;: 1794313225416,
    &quot;model_execution_start_ms&quot;: 1794313225391,
    &quot;model_load_end_ms&quot;: 1794313224426,
    &quot;model_load_start_ms&quot;: 1794313222794,
    &quot;prefill_token_per_sec&quot;: 232.143,
    &quot;prompt_eval_end_ms&quot;: 1794313224650,
    &quot;prompt_tokens&quot;: 52
  },
  &quot;operator_fallback_count&quot;: null,
  &quot;quality&quot;: null,
  &quot;quantization&quot;: &quot;8da4w: dynamic INT8 activations / INT4 linear weights; INT8 embeddings&quot;,
  &quot;raw_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-raw-5d7cafda.json&quot;,
  &quot;runtime&quot;: &quot;executorch&quot;,
  &quot;serving&quot;: null,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-c0fa578c.json&quot;,
  &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;single_request_seconds&quot;: 0.99,
  &quot;stage&quot;: &quot;run&quot;,
  &quot;startup_seconds&quot;: 1.632,
  &quot;status&quot;: &quot;completed&quot;,
  &quot;threads&quot;: 4,
  &quot;tokenizer_sha256&quot;: &quot;aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4&quot;,
  &quot;versions&quot;: {
    &quot;executorch&quot;: &quot;1.5.1&quot;,
    &quot;torch&quot;: &quot;2.15.0&quot;,
    &quot;torchao&quot;: &quot;0.18.0&quot;,
    &quot;transformers&quot;: &quot;5.22.0&quot;
  },
  &quot;wall_seconds&quot;: 32.393331750063226
}</pre>
</details>

In [9]:
sql_check = study.stage('sql_check')
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(sql_check)))


Starting edge stage: sql_check | session 20261110T121532-73d94917db


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| Result | completed | Yes | Yes | — | — | — | — | — |

| Session id | Session evidence path |
|---|---|
| 20261110T121532-73d94917db | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/sql_check-da715168.json |

<details><summary>Full result details</summary>

<pre>{
  &quot;base_snapshot_sha256&quot;: &quot;9d76380658964726b0ebbfd59e23fd141cee771d9a116197b9ea4caaa737b616&quot;,
  &quot;case_id&quot;: &quot;eval_001&quot;,
  &quot;database_sha256&quot;: &quot;f1a7100697b0980b56359632fde49d042fb147260da03931a53a3cbfa7f6830e&quot;,
  &quot;execution_success&quot;: true,
  &quot;generated_query&quot;: {
    &quot;columns&quot;: [
      &quot;treatment_arm&quot;,
      &quot;count&quot;
    ],
    &quot;error&quot;: null,
    &quot;ok&quot;: true,
    &quot;row_count&quot;: 4,
    &quot;rows&quot;: [
      {
        &quot;count&quot;: 48,
        &quot;treatment_arm&quot;: &quot;Investigational Drug Low Dose&quot;
      },
      {
        &quot;count&quot;: 44,
        &quot;treatment_arm&quot;: &quot;Placebo&quot;
      },
      {
        &quot;count&quot;: 40,
        &quot;treatment_arm&quot;: &quot;Investigational Drug High Dose&quot;
      },
      {
        &quot;count&quot;: 38,
        &quot;treatment_arm&quot;: &quot;Standard of Care&quot;
      }
    ],
    &quot;sql&quot;: &quot;SELECT * FROM (SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm) AS readonly_result LIMIT 100&quot;
  },
  &quot;matches_dell_base_snapshot&quot;: true,
  &quot;measured_at&quot;: &quot;2026-11-10T12:20:30.158312+00:00&quot;,
  &quot;mode&quot;: &quot;live&quot;,
  &quot;normalization&quot;: &quot;Removed only the terminal Qwen &lt;|im_end|&gt; token before SQL execution; raw output preserved.&quot;,
  &quot;question&quot;: &quot;How many subjects are Enrolled or Randomized by treatment arm?&quot;,
  &quot;raw_generated_text&quot;: &quot;SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm;&lt;|im_end|&gt;&quot;,
  &quot;recorded_dell_base_snapshot_sha256&quot;: &quot;9d76380658964726b0ebbfd59e23fd141cee771d9a116197b9ea4caaa737b616&quot;,
  &quot;reference_query&quot;: {
    &quot;columns&quot;: [
      &quot;treatment_arm&quot;,
      &quot;subject_count&quot;
    ],
    &quot;error&quot;: null,
    &quot;ok&quot;: true,
    &quot;row_count&quot;: 4,
    &quot;rows&quot;: [
      {
        &quot;subject_count&quot;: 40,
        &quot;treatment_arm&quot;: &quot;Investigational Drug High Dose&quot;
      },
      {
        &quot;subject_count&quot;: 48,
        &quot;treatment_arm&quot;: &quot;Investigational Drug Low Dose&quot;
      },
      {
        &quot;subject_count&quot;: 44,
        &quot;treatment_arm&quot;: &quot;Placebo&quot;
      },
      {
        &quot;subject_count&quot;: 38,
        &quot;treatment_arm&quot;: &quot;Standard of Care&quot;
      }
    ],
    &quot;sql&quot;: &quot;SELECT * FROM (SELECT treatment_arm, COUNT(*) AS subject_count FROM subjects WHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;) GROUP BY treatment_arm ORDER BY treatment_arm) AS readonly_result LIMIT 100&quot;
  },
  &quot;reference_sql&quot;: &quot;SELECT treatment_arm, COUNT(*) AS subject_count FROM subjects WHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;) GROUP BY treatment_arm ORDER BY treatment_arm&quot;,
  &quot;result_equivalence&quot;: true,
  &quot;scope&quot;: &quot;One direct short-schema SQL smoke; no plan, repair, full workload or human semantic review. SQL checks occur after native timing.&quot;,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/sql_check-da715168.json&quot;,
  &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;source&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-c0fa578c.json&quot;,
  &quot;source_sha256&quot;: &quot;887792b8d564d87058a62f0b16a04668ae441dd16e6223af568b41cffbd5e2e0&quot;,
  &quot;sql&quot;: &quot;SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm;&quot;,
  &quot;sql_sha256&quot;: &quot;20430b609bc2bab8ac6abfcb41f10b248079e4200b3a406dca03ccf5f1f729b2&quot;,
  &quot;status&quot;: &quot;completed&quot;
}</pre>
</details>

## 5. Read this session's measurements

Compare model loading, first-token time, inference latency, artifact footprint and process RSS. These are one short-question sample, not serving percentiles. RSS is CPU process memory, not GPU memory. Inspect generated SQL and its reference-result check alongside timing.


In [10]:
current = study.results.get('run', {})
if current.get('session_id') == study.session_id and current.get('status') == 'completed':
    observer = current.get('native_observer') or {}
    ttft = ((observer['first_token_ms'] - observer['inference_start_ms']) / 1000
            if 'first_token_ms' in observer and 'inference_start_ms' in observer else None)
    from clinical_trial_mvp.tutorial.result_display import result_table
    display(Markdown(result_table({'session': study.session_id,
        'model_load_seconds': current.get('startup_seconds'), 'ttft_seconds': ttft,
        'inference_seconds': current.get('single_request_seconds'),
        'footprint_bytes': current.get('footprint_bytes'), 'memory': current.get('memory'),
        'generation': current.get('generation'),
        'sql_check': study.results.get('sql_check')})))
else:
    print('Native measurements are unavailable until generation completes in this session.')
display(Markdown(study.report()))


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| Result | — | Yes | Yes | 0.224 | 0.990 | 1.632 | 4.908 | 2365.898 |

Memory scope: whole runner-process RSS, including Python/runtime imports; not GPU memory.

| Session |
|---|
| 20261110T121532-73d94917db |

<details><summary>Full result details</summary>

<pre>{
  &quot;footprint_bytes&quot;: 2480823486,
  &quot;generation&quot;: {
    &quot;num_generated_tokens&quot;: 29,
    &quot;num_prompt_tokens&quot;: 52,
    &quot;observer&quot;: {
      &quot;SCALING_FACTOR_UNITS_PER_SECOND&quot;: 1000,
      &quot;aggregate_model_execution_time_ms&quot;: 981,
      &quot;aggregate_sampling_time_ms&quot;: 8,
      &quot;decode_token_per_sec&quot;: 37.859,
      &quot;first_token_ms&quot;: 1794313224650,
      &quot;generated_tokens&quot;: 29,
      &quot;inference_end_ms&quot;: 1794313225416,
      &quot;inference_start_ms&quot;: 1794313224426,
      &quot;model_execution_end_ms&quot;: 1794313225416,
      &quot;model_execution_start_ms&quot;: 1794313225391,
      &quot;model_load_end_ms&quot;: 1794313224426,
      &quot;model_load_start_ms&quot;: 1794313222794,
      &quot;prefill_token_per_sec&quot;: 232.143,
      &quot;prompt_eval_end_ms&quot;: 1794313224650,
      &quot;prompt_tokens&quot;: 52
    },
    &quot;prompt&quot;: &quot;&lt;|im_start|&gt;user\nWrite only DuckDB SQL, without Markdown or explanation.\nTable subjects(subject_id VARCHAR PRIMARY KEY, treatment_arm VARCHAR, screening_status VARCHAR).\nHow many subjects are Enrolled or Randomized by treatment arm?&lt;|im_end|&gt;\n&lt;|im_start|&gt;assistant\n&lt;think&gt;\n\n&lt;/think&gt;\n\n&quot;,
    &quot;question&quot;: &quot;How many subjects are Enrolled or Randomized by treatment arm?&quot;,
    &quot;text&quot;: &quot;SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm;&lt;|im_end|&gt;&quot;
  },
  &quot;inference_seconds&quot;: 0.99,
  &quot;memory&quot;: {
    &quot;error&quot;: null,
    &quot;peak_bytes&quot;: 5269995520,
    &quot;sample_count&quot;: 297,
    &quot;sampling_interval_seconds&quot;: 0.1,
    &quot;scope&quot;: &quot;whole runner-process RSS, including Python/runtime imports; not GPU memory&quot;
  },
  &quot;model_load_seconds&quot;: 1.632,
  &quot;session&quot;: &quot;20261110T121532-73d94917db&quot;,
  &quot;sql_check&quot;: {
    &quot;base_snapshot_sha256&quot;: &quot;9d76380658964726b0ebbfd59e23fd141cee771d9a116197b9ea4caaa737b616&quot;,
    &quot;case_id&quot;: &quot;eval_001&quot;,
    &quot;database_sha256&quot;: &quot;f1a7100697b0980b56359632fde49d042fb147260da03931a53a3cbfa7f6830e&quot;,
    &quot;execution_success&quot;: true,
    &quot;generated_query&quot;: {
      &quot;columns&quot;: [
        &quot;treatment_arm&quot;,
        &quot;count&quot;
      ],
      &quot;error&quot;: null,
      &quot;ok&quot;: true,
      &quot;row_count&quot;: 4,
      &quot;rows&quot;: [
        {
          &quot;count&quot;: 48,
          &quot;treatment_arm&quot;: &quot;Investigational Drug Low Dose&quot;
        },
        {
          &quot;count&quot;: 44,
          &quot;treatment_arm&quot;: &quot;Placebo&quot;
        },
        {
          &quot;count&quot;: 40,
          &quot;treatment_arm&quot;: &quot;Investigational Drug High Dose&quot;
        },
        {
          &quot;count&quot;: 38,
          &quot;treatment_arm&quot;: &quot;Standard of Care&quot;
        }
      ],
      &quot;sql&quot;: &quot;SELECT * FROM (SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm) AS readonly_result LIMIT 100&quot;
    },
    &quot;matches_dell_base_snapshot&quot;: true,
    &quot;measured_at&quot;: &quot;2026-11-10T12:20:30.158312+00:00&quot;,
    &quot;mode&quot;: &quot;live&quot;,
    &quot;normalization&quot;: &quot;Removed only the terminal Qwen &lt;|im_end|&gt; token before SQL execution; raw output preserved.&quot;,
    &quot;question&quot;: &quot;How many subjects are Enrolled or Randomized by treatment arm?&quot;,
    &quot;raw_generated_text&quot;: &quot;SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm;&lt;|im_end|&gt;&quot;,
    &quot;recorded_dell_base_snapshot_sha256&quot;: &quot;9d76380658964726b0ebbfd59e23fd141cee771d9a116197b9ea4caaa737b616&quot;,
    &quot;reference_query&quot;: {
      &quot;columns&quot;: [
        &quot;treatment_arm&quot;,
        &quot;subject_count&quot;
      ],
      &quot;error&quot;: null,
      &quot;ok&quot;: true,
      &quot;row_count&quot;: 4,
      &quot;rows&quot;: [
        {
          &quot;subject_count&quot;: 40,
          &quot;treatment_arm&quot;: &quot;Investigational Drug High Dose&quot;
        },
        {
          &quot;subject_count&quot;: 48,
          &quot;treatment_arm&quot;: &quot;Investigational Drug Low Dose&quot;
        },
        {
          &quot;subject_count&quot;: 44,
          &quot;treatment_arm&quot;: &quot;Placebo&quot;
        },
        {
          &quot;subject_count&quot;: 38,
          &quot;treatment_arm&quot;: &quot;Standard of Care&quot;
        }
      ],
      &quot;sql&quot;: &quot;SELECT * FROM (SELECT treatment_arm, COUNT(*) AS subject_count FROM subjects WHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;) GROUP BY treatment_arm ORDER BY treatment_arm) AS readonly_result LIMIT 100&quot;
    },
    &quot;reference_sql&quot;: &quot;SELECT treatment_arm, COUNT(*) AS subject_count FROM subjects WHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;) GROUP BY treatment_arm ORDER BY treatment_arm&quot;,
    &quot;result_equivalence&quot;: true,
    &quot;scope&quot;: &quot;One direct short-schema SQL smoke; no plan, repair, full workload or human semantic review. SQL checks occur after native timing.&quot;,
    &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/sql_check-da715168.json&quot;,
    &quot;session_id&quot;: &quot;20261110T121532-73d94917db&quot;,
    &quot;source&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-c0fa578c.json&quot;,
    &quot;source_sha256&quot;: &quot;887792b8d564d87058a62f0b16a04668ae441dd16e6223af568b41cffbd5e2e0&quot;,
    &quot;sql&quot;: &quot;SELECT treatment_arm, COUNT(*) AS count\nFROM subjects\nWHERE screening_status IN (&#x27;Enrolled&#x27;, &#x27;Randomized&#x27;)\nGROUP BY treatment_arm;&quot;,
    &quot;sql_sha256&quot;: &quot;20430b609bc2bab8ac6abfcb41f10b248079e4200b3a406dca03ccf5f1f729b2&quot;,
    &quot;status&quot;: &quot;completed&quot;
  },
  &quot;ttft_seconds&quot;: 0.224
}</pre>
</details>

**Mode: live · Session: 20261110T121532-73d94917db**

| Stage | Status | Evidence |
|---|---|---|
| download | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/download-9d9f59a0.json |
| prepare | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/prepare-1d98828c.json |
| reference | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/reference-a29d110a.json |
| validate | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/validate-b6d0a0cb.json |
| export | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/export-69487e6c.json |
| run | completed | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-c0fa578c.json |
| sql_check | completed | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/sql_check-da715168.json |

## Cleanup

The edge stages execute in foreground child processes and retain no serving process. Save the notebook to retain genuine current outputs. Failures stay in the session evidence.


In [11]:
print(study.close())


Edge stages run in foreground child processes; no serving process is retained.


## Reference: previously measured results

The [edge guide](../tutorial/EXECUTORCH_MAC.md) preserves the previous Qwen native run and Nemotron compatibility check. Choose `MODE="replay"` explicitly to inspect those records. The saved artifacts and historical records remain unchanged; they are not results from a new Live execution.
